In [ ]:
import os, json, subprocess
import numpy as np

stats_dir = 'stats'
pdb_dir = '/mnt/hdd/yenlin/data/Protein_Data_Bank'
pdb_download_dir = f'{pdb_dir}/pdb.gz'
os.makedirs(pdb_dir, exist_ok=True)
os.makedirs(pdb_download_dir, exist_ok=True)


# PARAMETERS
coverage_mode = 0
# similarity_cutoff = 1
similarity_cutoff = 0.95

tag = f'sim_{similarity_cutoff}'


In [ ]:
### READ DATA FROM PREVIOUS SCRIPT

# META DATA
all_info = np.loadtxt(
    f'{stats_dir}/OUT-1-3.all_information.csv',
    delimiter=',',
    dtype=np.str_,
    comments=None
)

header = all_info[0].tolist()
header[0] = header[0][2:]
all_info = all_info[1:]

print(header)
print(len(all_info), 'entries')


['full_id', 'pdb_id', 'assembly_id', 'multimericity', 'auth_chain_id', 'asym_chain_id', 'seq_database', 'seq_database_accession']
38973 entries


In [3]:
# SIMILARITY CLUSTER WITH MMSEQS2 (INSTALLED IN CONDA ENVIRONMENT)

mmseqs2_output = f'OUT-2-1.cluster-sim_{similarity_cutoff}'
logfile = f'stats/OUT-2-1.mmseqs2-sim_{similarity_cutoff}.log'

cmd = [
    'mmseqs',
    'easy-cluster',
    f'{stats_dir}/OUT-1-3.all_sequences.fasta',
    f'{stats_dir}/{mmseqs2_output}',
    'tmp',
    '--min-seq-id', f'{similarity_cutoff}',
    '-c', '0.8',
    '--cov-mode', f'{coverage_mode}'
]

with open(logfile, 'w') as log:
    subprocess.run(cmd, stdout=log)



In [4]:
### PARSE CLUSTERING RESULTS

rep_ids = []
rep_seq = []
with open(f'{stats_dir}/{mmseqs2_output}_rep_seq.fasta', 'r') as f:
    for line in f.readlines():
        if line[0] == '>':
            rep_ids.append(line[1:].replace(' \n', '').split(',')[0])
        else:
            rep_seq.append(line.replace('\n', ''))

assert len(rep_seq) == len(rep_ids)

rep_ids = np.array(rep_ids)
print(len(rep_ids), 'clusters')


12534 clusters


In [5]:
### REMOVE IDENTICAL SEQUENCES (MIGHT OCCUR FOR SHORT SEQUENCES)

rep_seq_unique, count = np.unique(rep_seq, return_counts=True)
rep_ids_kept = rep_ids[count==1]
rep_seq_kept = rep_seq_unique[count==1]
rep_ids_removed = rep_ids[count>1]

print('# of duplicate sequences:', len(rep_ids_removed))
print(rep_ids_removed)

all_info_kept = all_info[
    np.isin(all_info[:,header.index('full_id')], rep_ids_kept)
]

# SUMMARY
print(f'{len(rep_ids)} sequences before removal')
print(f'{len(all_info_kept)} sequences after removal')
print()

_, cnt = np.unique(
    all_info_kept[:,header.index('multimericity')],
    return_counts=True
)
print(cnt, np.sum(cnt))

# of duplicate sequences: 0
[]
12534 sequences before removal
12534 sequences after removal

[9090 1006 1813   90  535] 12534


In [6]:
### SAVE INFO

np.savetxt(
    f'{stats_dir}/OUT-2-2.all_information-{tag}.csv',
    all_info_kept,
    delimiter=',',
    fmt='%s',
    header=','.join(header)
)

# FASTA
all_auth_chain_id_kept = all_info_kept[:,header.index('auth_chain_id')]
all_asym_chain_id_kept = all_info_kept[:,header.index('asym_chain_id')]
with open(f'{stats_dir}/OUT-2-2.all_sequences-{tag}.fasta', 'w') as f:
    for assembly_idx in range(len(rep_seq_kept)):
        f.write(
            f'>{rep_ids_kept[assembly_idx]}'
            f'auth_chain_id:{all_auth_chain_id_kept[assembly_idx]},'
            f'asym_chain_id:{all_asym_chain_id_kept[assembly_idx]}\n'
        )
        f.write(f'{rep_seq_kept[assembly_idx]}\n')


In [ ]:
### GENERATE LIST OF PDB TO DOWNLOAD

rep_pdb_ids_kept = np.unique(all_info_kept[:,header.index('pdb_id')])
print('Number of unique PDB IDs:', len(rep_pdb_ids_kept))

pdb_downloaded = [
    f[:4] for f in os.listdir(pdb_download_dir)
    if f.endswith('.pdb.gz')
]

pdb_to_download = rep_pdb_ids_kept[
    ~np.isin(rep_pdb_ids_kept, pdb_downloaded)
]
print('Number of PDBs yet downloaded:', len(pdb_to_download))

with open(f'{stats_dir}/OUT-temp.pdbs_to_download-{tag}.txt', 'w') as f:
    f.write(','.join(pdb_to_download) + '\n')

Number of unique PDB IDs: 12534
Number of PDBs yet downloaded: 125
